# Скачать MUSAN на Google Drive

Ночной Colab-ноутбук. Он работает только с ZIP по ссылке коллеги: скачивает его в твой Drive, извлекает настоящие шумы и удаляет ZIP для экономии места.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install gdown

Mounted at /content/drive


In [4]:
from pathlib import Path
import subprocess, tarfile, zipfile

DATA_ROOT = Path('/content/drive/MyDrive/audio-harness')
MUSAN_FILE_ID = '1kngNXIxSjXTV2JtS9PAbE03yQcnx7iu_'

DESTINATION = DATA_ROOT / 'raw' / 'musan'
DESTINATION.mkdir(parents=True, exist_ok=True)

ARCHIVE = DESTINATION / 'musan_archive'
legacy_archive = DESTINATION / 'musan.zip'

# Подхватит уже скачанный файл после прошлой попытки
if not ARCHIVE.exists() and legacy_archive.exists():
    legacy_archive.replace(ARCHIVE)

print('Шумы будут здесь:', DESTINATION)

Шумы будут здесь: /content/drive/MyDrive/audio-harness/raw/musan


In [5]:
noise_wavs = list(DESTINATION.rglob('noise/*.wav'))

if noise_wavs:
    print('Уже готово, WAV-шумов:', len(noise_wavs))
else:
    if not ARCHIVE.exists() or ARCHIVE.stat().st_size < 1_000_000_000:
        print('Скачиваю архив из Google Drive коллеги…')
        subprocess.run(
            ['gdown', '--fuzzy',
             f'https://drive.google.com/file/d/{MUSAN_FILE_ID}/view',
             '-O', str(ARCHIVE)],
            check=True
        )

    if tarfile.is_tarfile(ARCHIVE):
        print('Формат TAR/GZ. Достаю только noise…')
        with tarfile.open(ARCHIVE) as archive:
            members = [
                item for item in archive.getmembers()
                if '/noise/' in item.name and item.isfile()
            ]
            assert members, 'В архиве не найдена папка noise.'
            for item in members:
                archive.extract(item, DESTINATION)

    elif zipfile.is_zipfile(ARCHIVE):
        print('Формат ZIP. Достаю только noise…')
        with zipfile.ZipFile(ARCHIVE) as archive:
            members = [
                name for name in archive.namelist()
                if '/noise/' in name and not name.endswith('/')
            ]
            assert members, 'В архиве не найдена папка noise.'
            for item in members:
                archive.extract(item, DESTINATION)
    else:
        raise RuntimeError('Архив не распознан.')

    ARCHIVE.unlink()
    print('Готово, WAV-шумов:', len(list(DESTINATION.rglob('noise/*.wav'))))

Формат TAR/GZ. Достаю только noise…


/tmp/ipykernel_5356/2694975932.py:24: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extract(item, DESTINATION)


Готово, WAV-шумов: 0


После этого чистую русскую речь берём из твоего локального `C:\Users\Vadim\projects\pps2k26\data\ru_RU.tgz`. Её не нужно дублировать в Colab, пока не начнём именно обучение в Colab.

In [6]:
wav_files = list(DESTINATION.rglob('*.wav'))

print('Всего WAV:', len(wav_files))
for path in wav_files[:5]:
    print(path)

Всего WAV: 930
/content/drive/MyDrive/audio-harness/raw/musan/musan/noise/free-sound/noise-free-sound-0001.wav
/content/drive/MyDrive/audio-harness/raw/musan/musan/noise/free-sound/noise-free-sound-0002.wav
/content/drive/MyDrive/audio-harness/raw/musan/musan/noise/free-sound/noise-free-sound-0003.wav
/content/drive/MyDrive/audio-harness/raw/musan/musan/noise/free-sound/noise-free-sound-0004.wav
/content/drive/MyDrive/audio-harness/raw/musan/musan/noise/free-sound/noise-free-sound-0005.wav


In [9]:

DATA_ROOT = Path('/content/drive/MyDrive/audio-harness')
MAILABS_DIR = DATA_ROOT / 'raw' / 'm_ailabs_ru'

# Ищем загруженный архив, даже если он оказался не в raw/
archives = list(Path('/content/drive/MyDrive').rglob('ru_RU.tgz'))
assert archives, 'ru_RU.tgz не найден в Google Drive.'

MAILABS_ARCHIVE = archives[0]
print('Архив:', MAILABS_ARCHIVE)
print('Размер, GB:', round(MAILABS_ARCHIVE.stat().st_size / 1024**3, 2))

if not (MAILABS_DIR / 'ru_RU' / 'by_book').exists():
    MAILABS_DIR.mkdir(parents=True, exist_ok=True)
    print('Распаковываю M-AILABS…')
    subprocess.run(
        ['tar', '-xzf', str(MAILABS_ARCHIVE), '-C', str(MAILABS_DIR)],
        check=True
    )

wav_files = list(MAILABS_DIR.rglob('*.wav'))
print('Готово. WAV-файлов:', len(wav_files))
print('Пример:', wav_files[0] if wav_files else 'не найдено')

Архив: /content/drive/MyDrive/audio-harness/raw/ru_RU.tgz
Размер, GB: 3.6
Распаковываю M-AILABS…
Готово. WAV-файлов: 20499
Пример: /content/drive/MyDrive/audio-harness/raw/m_ailabs_ru/ru_RU/by_book/female/hajdurova/chetvero_nischih/wavs/chetvero_nischih_s001052.wav


In [10]:
!pip -q install soundfile

import csv
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import soundfile as sf

SEED = 42
PAIR_COUNT = 100

MAILABS_DIR = DATA_ROOT / 'raw' / 'm_ailabs_ru'
NOISE_DIR = DATA_ROOT / 'raw' / 'musan' / 'musan' / 'noise'
OUT = DATA_ROOT / 'derived' / 'denoise_eval'

rng = random.Random(SEED)

# Берём целую книгу как test, чтобы клипы этой книги не смешивались с train позже.
books = defaultdict(list)
for wav in MAILABS_DIR.rglob('*.wav'):
    rel = wav.relative_to(MAILABS_DIR)
    parts = rel.parts
    marker = parts.index('by_book')
    key = tuple(parts[marker + 1:marker + 4])  # gender, speaker, book
    books[key].append(wav)

speaker_books = defaultdict(list)
for key in books:
    speaker_books[key[:2]].append(key)

candidates = [
    book for speaker, speaker_book_list in speaker_books.items()
    if len(speaker_book_list) > 1
    for book in speaker_book_list
]
rng.shuffle(candidates)
test_book = candidates[1]
clean_files = books[test_book]

noise_files = list(NOISE_DIR.rglob('*.wav'))
assert noise_files, f'Не найдены шумы в {NOISE_DIR}'

print('Test-книга:', '/'.join(test_book))
print('Чистых клипов в ней:', len(clean_files))
print('MUSAN-шумов:', len(noise_files))

for folder in ('clean', 'noise', 'noisy'):
    (OUT / folder).mkdir(parents=True, exist_ok=True)

def mono(audio):
    return audio.mean(axis=1) if audio.ndim == 2 else audio

def resample_linear(audio, old_rate, new_rate):
    if old_rate == new_rate:
        return audio
    new_length = round(len(audio) * new_rate / old_rate)
    return np.interp(
        np.linspace(0, 1, new_length, endpoint=False),
        np.linspace(0, 1, len(audio), endpoint=False),
        audio
    ).astype(np.float32)

def fit_noise(audio, length):
    if len(audio) >= length:
        start = rng.randrange(len(audio) - length + 1)
        return audio[start:start + length]
    return np.tile(audio, int(np.ceil(length / len(audio))))[:length]

def rms(audio):
    return float(np.sqrt(np.mean(audio.astype(np.float64) ** 2)))

rng.shuffle(clean_files)
snrs = ([0, 5, 10] * ((PAIR_COUNT + 2) // 3))[:PAIR_COUNT]
rng.shuffle(snrs)

manifest = OUT / 'manifest.csv'
with manifest.open('w', newline='', encoding='utf-8') as f:
    fields = ['id', 'clean', 'noise', 'noisy', 'snr_db', 'source_clean', 'source_noise']
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()

    for i, (clean_path, snr) in enumerate(zip(clean_files[:PAIR_COUNT], snrs)):
        clean, sample_rate = sf.read(clean_path, dtype='float32', always_2d=False)
        clean = mono(clean)

        noise_path = rng.choice(noise_files)
        noise, noise_rate = sf.read(noise_path, dtype='float32', always_2d=False)
        noise = fit_noise(resample_linear(mono(noise), noise_rate, sample_rate), len(clean))

        noise *= rms(clean) / max(rms(noise), 1e-8) / (10 ** (snr / 20))
        noisy = clean + noise

        peak = max(float(np.max(np.abs(noisy))), 1.0)
        clean, noise, noisy = clean / peak, noise / peak, noisy / peak

        item_id = f'{i:04d}_snr{snr}'
        clean_out = f'clean/{item_id}.wav'
        noise_out = f'noise/{item_id}.wav'
        noisy_out = f'noisy/{item_id}.wav'

        sf.write(OUT / clean_out, clean, sample_rate, subtype='PCM_16')
        sf.write(OUT / noise_out, noise, sample_rate, subtype='PCM_16')
        sf.write(OUT / noisy_out, noisy, sample_rate, subtype='PCM_16')

        writer.writerow({
            'id': item_id,
            'clean': clean_out,
            'noise': noise_out,
            'noisy': noisy_out,
            'snr_db': snr,
            'source_clean': str(clean_path.relative_to(MAILABS_DIR)),
            'source_noise': str(noise_path.relative_to(NOISE_DIR)),
        })

print('Готово:', manifest)

Test-книга: female/hajdurova/mnogo_shuma_iz_nichego
Чистых клипов в ней: 2387
MUSAN-шумов: 930
Готово: /content/drive/MyDrive/audio-harness/derived/denoise_eval/manifest.csv


In [14]:
!uv pip install \
    --python /content/df311/bin/python \
    torch==2.3.1 torchaudio==2.3.1 \
    --index-url https://download.pytorch.org/whl/cpu \
    --extra-index-url https://pypi.org/simple

!uv pip install \
    --python /content/df311/bin/python \
    DeepFilterNet==0.5.6 soundfile

!/content/df311/bin/python -c "import torch; import df; print('OK:', torch.__version__)"

Using Python 3.11.17 environment at: df311
Resolved 23 packages in 855ms
Prepared 23 packages in 1m 17s
Installed 23 packages in 270ms
 + filelock==4.2.4
 + fsspec==2026.9.0
 + jinja2==3.1.6
 + markupsafe==3.0.4
 + mpmath==1.3.0
 + networkx==3.6.1
 + nvidia-cublas-cu12==12.1.3.1
 + nvidia-cuda-cupti-cu12==12.1.105
 + nvidia-cuda-nvrtc-cu12==12.1.105
 + nvidia-cuda-runtime-cu12==12.1.105
 + nvidia-cudnn-cu12==8.9.2.26
 + nvidia-cufft-cu12==11.0.2.54
 + nvidia-curand-cu12==10.3.2.106
 + nvidia-cusolver-cu12==11.4.5.107
 + nvidia-cusparse-cu12==12.1.0.106
 + nvidia-nccl-cu12==2.20.5
 + nvidia-nvjitlink-cu12==12.9.86
 + nvidia-nvtx-cu12==12.1.105
 + sympy==1.14.0
 + torch==2.3.1
 + torchaudio==2.3.1
 + triton==2.3.1
 + typing-extensions==4.16.0
Using Python 3.11.17 environment at: df311
Resolved 17 packages in 270ms
Prepared 14 packages in 523ms
Installed 14 packages in 26ms
 + appdirs==1.4.4
 + certifi==2026.10.19
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + deepfilterlib==0.5.6
 + deep

In [15]:
!pip -q install DeepFilterNet

import csv
import sys
import types
from pathlib import Path

try:
    from torchaudio.backend.common import AudioMetaData
except ModuleNotFoundError:
    compat = types.ModuleType("torchaudio.backend.common")
    compat.AudioMetaData = object
    sys.modules["torchaudio.backend.common"] = compat

from df.enhance import enhance, init_df, load_audio, save_audio
from IPython.display import Audio, display

model, df_state, _ = init_df()

with (OUT / 'manifest.csv').open(encoding='utf-8') as f:
    item = next(csv.DictReader(f))

noisy_path = OUT / item['noisy']
denoised_path = OUT / 'denoised_0000.wav'

audio, _ = load_audio(noisy_path, sr=df_state.sr())
denoised = enhance(model, df_state, audio)
save_audio(denoised_path, denoised, df_state.sr())

print('SNR:', item['snr_db'], 'dB')
print('Сохранено:', denoised_path)

print('\nClean:')
display(Audio(str(OUT / item['clean'])))
print('Noisy:')
display(Audio(str(noisy_path)))
print('Denoised:')
display(Audio(str(denoised_path)))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


ModuleNotFoundError: No module named 'df'